# Week 6: Advanced Retrieval, Transform and Measure

### Setup: Week 5 pipeline

Copied from `05-rag-and-retrieval/week5_rag_pipeline.ipynb`. **Corpus:** eight IETF RFCs, loaded from `rfcs/` (fetched from rfc-editor.org if missing). Only the by-paragraph chunking config is built.

In [1]:
import os, re, pathlib, urllib.request, numpy as np
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from collections import Counter
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

RFCS = {8259: 'JSON', 3339: 'Date and time', 4648: 'Base16/32/64 encodings', 7519: 'JSON Web Token',
        6750: 'OAuth 2.0 bearer tokens', 6265: 'HTTP cookies', 6455: 'WebSocket', 9562: 'UUIDs'}
RFC_DIR = pathlib.Path('rfcs'); RFC_DIR.mkdir(exist_ok=True)

def load_rfc(n):
    path = RFC_DIR / f'rfc{n}.txt'
    if not path.exists():
        urllib.request.urlretrieve(f'https://www.rfc-editor.org/rfc/rfc{n}.txt', path)
    text = path.read_text(encoding='utf-8')
    # Page break = footer line, form feed, running header (absent after the last page).
    return re.sub(r'\n[^\n]*\[Page \d+\]\n\f\n(?:RFC \d+[^\n]*\n)?', '\n', text)

CORPUS = {n: load_rfc(n) for n in RFCS}
for n, text in CORPUS.items():
    print(f'RFC {n}  {RFCS[n]:24s} {len(text):7,d} chars')
print('corpus chars:', sum(map(len, CORPUS.values())))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RFC 8259  JSON                      26,065 chars
RFC 3339  Date and time             32,473 chars
RFC 4648  Base16/32/64 encodings    32,900 chars
RFC 7519  JSON Web Token            58,672 chars
RFC 6750  OAuth 2.0 bearer tokens   36,358 chars
RFC 6265  HTTP cookies              74,321 chars
RFC 6455  WebSocket                151,632 chars
RFC 9562  UUIDs                    114,636 chars
corpus chars: 527057


In [2]:
def chunk_paragraph(text):
    return [re.sub(r'\s+',' ',p).strip() for p in text.split('\n\n') if p.strip()]
def chunk_corpus(chunker, *args):
    # Chunk each RFC separately so no chunk spans two documents; tag each chunk with its RFC.
    return [(n, c) for n, text in CORPUS.items() for c in chunker(text, *args)]

def build(chunks):
    e = embedder.encode([t for _, t in chunks], normalize_embeddings=True).astype('float32')
    idx = faiss.IndexFlatIP(e.shape[1]); idx.add(e); return idx
def search(idx, q, k=3):
    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')
    # FAISS uses -1 for missing results when fewer than k chunks exist.
    return [i for i in idx.search(qe, k)[1][0] if i >= 0]

chunks = chunk_corpus(chunk_paragraph)
index = build(chunks)
print(f'by-paragraph -> {len(chunks)} chunks')

by-paragraph -> 2604 chunks


In [3]:
# (query, relevant RFC, acceptable answer strings)
qa = [
    ('what text encoding must JSON use when exchanged between systems', 8259,
     ('MUST be encoded using UTF-8', 'require the use of UTF-8 when transmitted')),
    ('what character separates the date from the time in an internet timestamp', 3339,
     ('full-date "T" full-time', 'date and time separated by "T"', 'requires the "T"')),
    ('which character replaces the plus sign in URL-safe base64', 4648,
     ('62 - (minus)',)),
    ('which claim stops a JWT from being used before a certain time', 7519,
     ('the time before which the JWT MUST NOT be accepted',)),
    ('what status code should a server return for an expired access token', 6750,
     ('HTTP 401 (Unauthorized)', 'expired access token: HTTP/1.1 401')),
    ('response when the token lacks the permissions a request needs', 6750,
     ('403 (Forbidden)',)),
    ('if a cookie sets both Max-Age and Expires, which one wins', 6265,
     ('attribute has precedence',)),
    ('which opcode is used for a websocket ping frame', 6455,
     ('%x9 denotes a ping', '%x9 ; ping', '0x9 (Ping)', 'opcode of 0x9')),
    ('what protocol version does the client send in the websocket opening handshake', 6455,
     ('with a value of 13', 'MUST be 13', 'Sec-WebSocket-Version: 13')),
    ('how many bits of a UUIDv7 hold the millisecond timestamp', 9562,
     ('most significant 48 bits', '48-bit big-endian unsigned number of the Unix Epoch',
      'UUIDv7 features a 48-bit timestamp', 'millisecond precision to fill the first 48 bits')),
]
norm = lambda s: re.sub(r'\s+', ' ', s).lower()
missing = [f for _, n, facts in qa for f in facts if norm(f) not in norm(CORPUS[n])]
leaks = [(f, d) for _, n, facts in qa for f in facts for d in CORPUS if d != n and norm(f) in norm(CORPUS[d])]
assert not missing and not leaks, (missing, leaks)
print(len(qa), 'queries; every answer string found only in its own RFC')

10 queries; every answer string found only in its own RFC


**Metrics** (same definitions as Week 5, plus two additions)
- **P@3**: the share of the 3 retrieved chunks that are relevant (contain an answer string).
- **R@3**: the share of a query's relevant chunks that were retrieved.
- **hit@3**: whether at least one relevant chunk was retrieved.
- **doc-P@3**: the share of retrieved chunks that come from the correct RFC.
- **MRR@3**: mean reciprocal rank of the first relevant chunk (1, 1/2, 1/3, or 0). Re-ranking can reorder the top 3 without changing hit@3; MRR@3 makes that visible.
- **pool@20** (re-rank conditions only): whether any relevant chunk is among the 20 candidates handed to the re-ranker. A re-ranker can only promote what retrieval already found.

In [4]:
def relevant(chunks, rfc, facts):
    return {i for i, (n, t) in enumerate(chunks) if n == rfc and any(norm(f) in norm(t) for f in facts)}
# Relevance labels are fixed from the chunk text alone, before any retrieval runs.
labels = [relevant(chunks, rfc, facts) for _, rfc, facts in qa]

K = 3
METRICS = ['P@3', 'R@3', 'hit@3', 'doc-P@3', 'MRR@3']
def first_rank(ids, rel):
    return next((r for r, i in enumerate(ids, 1) if i in rel), None)
def evaluate(lists):
    rows = list(zip(lists, labels, qa))
    return {'P@3': np.mean([len(set(ids) & rel) / K for ids, rel, _ in rows]),
            # No relevant chunk means chunking split every occurrence of the answer; score recall 0.
            'R@3': np.mean([len(set(ids) & rel) / len(rel) if rel else 0 for ids, rel, _ in rows]),
            'hit@3': np.mean([bool(set(ids) & rel) for ids, rel, _ in rows]),
            'doc-P@3': np.mean([sum(chunks[i][0] == rfc for i in ids) / K for ids, _, (_, rfc, _) in rows]),
            'MRR@3': np.mean([1 / r if (r := first_rank(ids, rel)) else 0 for ids, rel, _ in rows])}

base = [search(index, q, K) for q, _, _ in qa]
print('relevant chunks per query:', [len(r) for r in labels])
print(f'\n{"condition":12s}' + ''.join(f'{m:>9s}' for m in METRICS))
print(f'{"baseline":12s}' + ''.join(f'{v:9.2f}' for v in evaluate(base).values()))

relevant chunks per query: [2, 3, 1, 1, 1, 1, 1, 4, 7, 4]

condition         P@3      R@3    hit@3  doc-P@3    MRR@3
baseline         0.27     0.53     0.70     0.97     0.55


### Part 1 Transform and Rerank

#### Query transformation

PRF assumes the top-ranked chunks for the original query are relevant, and appends their most frequent content words to the query before retrieving again. Here the top `feedback_k=3` chunks are read and `add=4` terms are appended (stopwords, words of 3 letters or fewer, and words already in the query are skipped). The expansion terms for every query are printed so the effect of each expansion can be traced.

In [5]:
STOP = set('a an the of to in on for and or is are how do does what so that with by as it why'.split())
def prf_expand(q, feedback_k=3, add=4):
    words = re.findall(r'[a-zA-Z]+', ' '.join(chunks[i][1] for i in search(index, q, feedback_k)).lower())
    cand = [w for w in words if w not in STOP and len(w) > 3 and w not in q.lower()]
    return q + ' ' + ' '.join(w for w, _ in Counter(cand).most_common(add))

expanded = [prf_expand(q) for q, _, _ in qa]
prf = [search(index, e, K) for e in expanded]
print('PRF expansion terms:')
for (q, _, _), e in zip(qa, expanded):
    print(f'  {q[:55]:56s} + {e[len(q) + 1:]}')

PRF expansion terms:
  what text encoding must JSON use when exchanged between  + have however software implementations
  what character separates the date from the time in an i  + timestamps this there many
  which character replaces the plus sign in URL-safe base  + encoding filename alphabet this
  which claim stops a JWT from being used before a certai  + must processing date expiration
  what status code should a server return for an expired   + resource request invalid protected
  response when the token lacks the permissions a request  + access scope resource protected
  if a cookie sets both Max-Age and Expires, which one wi  + attribute user agent date
  which opcode is used for a websocket ping frame          + denotes type defined section
  what protocol version does the client send in the webso  + http server clients talking
  how many bits of a UUIDv7 hold the millisecond timestam  + value this fraction values


#### Re-ranking: cross-encoder over retrieval candidates

The MiniLM bi-encoder embeds the query and each chunk separately and compares vectors. A cross-encoder (`cross-encoder/ms-marco-MiniLM-L6-v2`) reads the query and a chunk together and outputs a relevance score; it is more accurate but too slow to run over all chunks. So FAISS over-retrieves `N=20` candidates and the cross-encoder reorders them, keeping the top 3.

Two re-rank conditions are run:
- **rerank**: candidates retrieved with the original query.
- **PRF+rerank**: candidates retrieved with the PRF-expanded query, but scored by the cross-encoder against the *original* query. The expansion only changes which candidates enter the pool; relevance is judged against what was actually asked.

In [6]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')
N = 20
def rerank(q, cand):
    scores = reranker.predict([(q, chunks[i][1]) for i in cand])
    order = np.argsort(scores)[::-1]
    return [cand[j] for j in order], scores[order]

pools = {'rerank': [search(index, q, N) for q, _, _ in qa],
         'PRF+rerank': [search(index, e, N) for e in expanded]}
# Full reordered pool and scores are kept for the failure investigation; only the top K is evaluated.
reranked = {name: [rerank(q, cand) for (q, _, _), cand in zip(qa, pool)] for name, pool in pools.items()}

runs = {'baseline': base, 'PRF': prf,
        'rerank': [ids[:K] for ids, _ in reranked['rerank']],
        'PRF+rerank': [ids[:K] for ids, _ in reranked['PRF+rerank']]}
pool_hit = {name: np.mean([bool(set(cand) & rel) for cand, rel in zip(pool, labels)]) for name, pool in pools.items()}
for name, v in pool_hit.items():
    print(f'{name:12s} pool@{N}: {v:.2f}')

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

rerank       pool@20: 0.80
PRF+rerank   pool@20: 0.80


### Part 2 Measurement

Comparison of before and after transformation and reranking.

In [7]:
results = {name: evaluate(lists) for name, lists in runs.items()}
print(f'{"condition":12s}' + ''.join(f'{m:>9s}' for m in METRICS) + f'{"pool@20":>9s}')
for name, m in results.items():
    pool = f'{pool_hit[name]:9.2f}' if name in pool_hit else f'{"n/a":>9s}'
    print(f'{name:12s}' + ''.join(f'{m[k]:9.2f}' for k in METRICS) + pool)
print('\ndelta vs baseline')
for name, m in results.items():
    if name != 'baseline':
        print(f'{name:12s}' + ''.join(f'{m[k] - results["baseline"][k]:+9.2f}' for k in METRICS))

ranks = {name: [first_rank(ids, rel) for ids, rel in zip(lists, labels)] for name, lists in runs.items()}
rr = lambda r: 1 / r if r else 0
def flag(name, j):
    a, b = rr(ranks[name][j]), rr(ranks['baseline'][j])
    return 'IMPROVED' if a > b else ('REGRESSED' if a < b else 'same')
ARROW = {'IMPROVED': ' ↑', 'REGRESSED': ' ↓', 'same': '  '}
CONDS = [n for n in runs if n != 'baseline']

print(f'\n{"query":46s}' + ''.join(f'{n:>13s}' for n in runs))
for j, (q, _, _) in enumerate(qa):
    cells = [f'{ranks[n][j] or "-"} ({len(set(runs[n][j]) & labels[j])})' + ('  ' if n == 'baseline' else ARROW[flag(n, j)])
             for n in runs]
    print(f'{q[:45]:46s}' + ''.join(f'{c:>13s}' for c in cells))

print()
for name in CONDS:
    flags = [flag(name, j) for j in range(len(qa))]
    print(f'{name:12s} improved {flags.count("IMPROVED")}, regressed {flags.count("REGRESSED")}, same {flags.count("same")}')

condition         P@3      R@3    hit@3  doc-P@3    MRR@3  pool@20
baseline         0.27     0.53     0.70     0.97     0.55      n/a
PRF              0.20     0.47     0.60     0.97     0.45      n/a
rerank           0.33     0.58     0.80     0.90     0.80     0.80
PRF+rerank       0.33     0.58     0.80     0.90     0.80     0.80

delta vs baseline
PRF             -0.07    -0.05    -0.10    +0.00    -0.10
rerank          +0.07    +0.06    +0.10    -0.07    +0.25
PRF+rerank      +0.07    +0.06    +0.10    -0.07    +0.25

query                                              baseline          PRF       rerank   PRF+rerank
what text encoding must JSON use when exchang       1 (1)        2 (1) ↓      1 (1)        1 (1)  
what character separates the date from the ti       - (0)        - (0)        1 (1) ↑      1 (1) ↑
which character replaces the plus sign in URL       - (0)        - (0)        - (0)        - (0)  
which claim stops a JWT from being used befor       2 (1)        2 (1)     

PRF only improved one query and made three worse it clearly shows some regression likely because the expansion terms are mostly generic works like: have, however, this, there, many. The most notable result is that re-ranking made a much bigger difference than PRF it improved 4 queries and made none worse and once it was added to the process PRF essentially wasn't adding anything of value. It's clear that PRF made the rankings worse and  re-ranking actually improved them in this specific example.

### Part 3 Failure Investigation

**Re-ranker pool positions.** For each query and re-rank condition: the position of the first relevant chunk in the FAISS candidate list → its position after cross-encoder re-ranking, with its cross-encoder score next to the score of the 3rd kept chunk (the cut-off it had to beat). `not in pool` means no relevant chunk was among the 20 candidates.

In [8]:
def pool_line(name, j):
    ids, scores = reranked[name][j]
    before, after = first_rank(pools[name][j], labels[j]), first_rank(ids, labels[j])
    if before is None:
        return 'not in pool'
    return f'{before:>2d} -> {after:>2d}  score {scores[after - 1]:+6.2f} vs cut-off {scores[K - 1]:+6.2f}'

print(f'{"query":46s}' + ''.join(f'{n:>42s}' for n in pools))
for j, (q, _, _) in enumerate(qa):
    print(f'{q[:45]:46s}' + ''.join(f'{pool_line(n, j):>42s}' for n in pools))

query                                                                             rerank                                PRF+rerank
what text encoding must JSON use when exchang    1 ->  1  score  +8.84 vs cut-off  +3.32   2 ->  1  score  +8.84 vs cut-off  +3.32
what character separates the date from the ti   11 ->  1  score  +5.50 vs cut-off  +0.22  13 ->  1  score  +5.50 vs cut-off  +0.22
which character replaces the plus sign in URL                                not in pool                               not in pool
which claim stops a JWT from being used befor    2 ->  1  score  +6.94 vs cut-off  +4.61   2 ->  1  score  +6.94 vs cut-off  +4.61
what status code should a server return for a    1 ->  1  score  +7.31 vs cut-off  +4.40   1 ->  1  score  +7.31 vs cut-off  +4.40
response when the token lacks the permissions    2 ->  1  score  +2.37 vs cut-off  +0.81   1 ->  1  score  +2.37 vs cut-off  +0.81
if a cookie sets both Max-Age and Expires, wh    1 ->  1  score  +6.78 vs cut-off  

**Changed queries in detail.** For each query whose first-relevant rank changed under any condition: the original and PRF-expanded query, the baseline top 3, and the top 3 of each condition that changed it. `*` marks a relevant chunk (contains an answer string).

In [9]:
def show(ids, rel):
    for i in ids:
        print(f'      {"*" if i in rel else " "} [RFC {chunks[i][0]}] {chunks[i][1][:110]}')

for j, (q, _, _) in enumerate(qa):
    changed = [n for n in CONDS if flag(n, j) != 'same']
    if not changed:
        continue
    print(f'query:    {q}')
    print(f'expanded: {expanded[j]}')
    print(f'  baseline (first relevant rank {ranks["baseline"][j] or "-"}):')
    show(runs['baseline'][j], labels[j])
    for n in changed:
        print(f'  {n} {flag(n, j)} (first relevant rank {ranks[n][j] or "-"}):')
        show(runs[n][j], labels[j])
        if n in pools:
            print(f'      pool: {pool_line(n, j)}')
    print()

query:    what text encoding must JSON use when exchanged between systems
expanded: what text encoding must JSON use when exchanged between systems have however software implementations
  baseline (first relevant rank 1):
      * [RFC 8259] JSON text exchanged between systems that are not part of a closed ecosystem MUST be encoded using UTF-8 [RFC36
        [RFC 8259] Previous specifications of JSON have not required the use of UTF-8 when transmitting JSON text. However, the v
        [RFC 8259] When all the strings represented in a JSON text are composed entirely of Unicode characters [UNICODE] (however
  PRF REGRESSED (first relevant rank 2):
        [RFC 8259] Previous specifications of JSON have not required the use of UTF-8 when transmitting JSON text. However, the v
      * [RFC 8259] JSON text exchanged between systems that are not part of a closed ecosystem MUST be encoded using UTF-8 [RFC36
        [RFC 8259] When all the strings represented in a JSON text are composed entirel

Looking at the query: "which opcode is used for a websocket ping frame"; PRF regresses the retrieval results and re-ranking improves them. PRF makes the assumption that adding words from highly ranked results to the transformed query will improve the results, but when the expansion words weaken the important signal in the query they can clearly have the opposite effect.